In [1]:
!git clone https://github.com/paulphilip-louis/interpreting-prompt-injection.git

Cloning into 'interpreting-prompt-injection'...
remote: Enumerating objects: 91, done.
remote: Counting objects: 100% (91/91), done.
remote: Compressing objects: 100% (65/65), done.
remote: Total 91 (delta 37), reused 69 (delta 22), pack-reused 0 (from 0)
Receiving objects: 100% (91/91), 1.53 MiB | 13.65 MiB/s, done.
Resolving deltas: 100% (37/37), done.


In [2]:
import os
os.chdir("/content/interpreting-prompt-injection")

In [3]:
%pip install transformer_lens==2.15.0
%pip install transformers==4.51.0

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 189.2/189.2 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.7/739.7 kB 58.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 7.2 MB/s eta 0:00:00
  Created wheel for transformers-stream-generator: filename=transformers_stream_generator-0.0.5-py3-none-any.whl size=12426 sha256=1f82f147b87f6cd5b2570de4c18626fcfc9859863becc7494df9798c0785a5b6
  Stored in directory: /root/.cache/pip/wheels/a8/58/d2/014cb67c3cc6def738c1b1635dbf4e3dab6fb63aba7070dce0
Successfully built transformers-stream-generator
  Attempting uninstall: beartype
    Found existing installation: beartype 0.22.9
    Uninstalling beartype-0.22.9:
      Successfully uninstalled beartype-0.22.9
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
plum-dispatch 2.8.0 requires 

In [4]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import torch
from transformer_lens import HookedTransformer, ActivationCache
from tqdm import tqdm
import einops
import src.data.opi as opi
import src.utils.utils as utils

In [5]:
from matplotlib.pyplot import imshow

In [6]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

TASK_TYPE = "sentiment"
INJECTED_TASK = "spam"

In [7]:
model = HookedTransformer.from_pretrained(MODEL_NAME, device=DEVICE)
tokenizer = model.tokenizer

prompts = opi.load_opi(model=model, task_type=TASK_TYPE, injected_task=INJECTED_TASK, include_clean=True)
safe, naive, escape, ignore, combine = prompts.values()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

...

Generating train split:   0%|          | 0/33600 [00:00<?, ? examples/s]

Filter:   0%|          | 0/33600 [00:00<?, ? examples/s]

Filter:   0%|          | 0/33600 [00:00<?, ? examples/s]

Filter:   0%|          | 0/33600 [00:00<?, ? examples/s]

Filter:   0%|          | 0/33600 [00:00<?, ? examples/s]

In [8]:
# Useful values
n_layers = model.cfg.n_layers
n_heads = model.cfg.n_heads
pattern_layers = [f"blocks.{i}.attn.hook_pattern" for i in range(n_layers)]
CONDITIONS = ["safe", "naive", "escape", "ignore", "combine"]
ANSWER_STRINGS = {
    "spam":["spam", "Spam", "not spam", "Not spam", "SPAM", "NOT SPAM"],
    "hsol":["yes", "no", "Yes", "No", "YES", "NO"],
    "sentiment":["positive", "negative", "Positive", "Negative", "POSITIVE", "NEGATIVE"],
    "rte":["entailment", "not entailment", "Entailment", "Not entailment", "ENTAILMENT", "NOT ENTAILMENT"],
    "mrpc":["equivalent", "not equivalent", "Equivalent", "NOT EQUIVALENT", "EQUIVALENT", "Not equivalent"]
}

In [9]:
neurexec_pre = "(..']?></<|endoftext|>WithContext/helpensitivityuai/schemaaksiestionchtsInteractionEnabledunfinishedhelloarten"
neurexec_post = '.Can.Disclaimer":["NSMutableArray(_:'

random_pre = "Generic.ResultsVARCHARoczgregarInitialregionsASSWORDFc-volumePointSize,responseolphlidizin"
random_post = "Anal_results=valueMMutar"

neural_exec = []
for prompt in naive:
  # Specific for spam task
  idx1 = prompt.find("Write spam")
  idx2 = prompt.find("\n<|im_end|>\n<|im_start|>assistant")
  neural_exec.append(prompt[:idx1] + neurexec_pre + prompt[idx1:idx2] + neurexec_post + prompt[idx2:])

random = []
for prompt in naive:
  idx1 = prompt.find("Write spam")
  idx2 = prompt.find("\n<|im_end|>\n<|im_start|>assistant")
  random.append(prompt[:idx1] + random_pre + prompt[idx1:idx2] + random_post + prompt[idx2:])

# Attribution patching

The core idea of attribution patching is using a first-order approximation for activation patching rather than recomputing everything.

We start experiments by patching from naive triggers to combine triggers:

In [10]:
clean = naive[:2]
corrupted = combine[:2]

In [11]:
def run_all_prompts_with_cache(prompts, names_filter=None):
    logits_batch = []
    cache_batch = {}
    device = next(model.parameters()).device
    tokens = model.to_tokens(prompts)
    for prompt in tqdm(tokens):
        prompt = prompt.to(device)
        logits, cache = model.run_with_cache(prompt, names_filter=names_filter)
        logits_batch.append(logits)
        if cache_batch == {}:
            cache_batch = cache
        else:
            for k,v in cache.items():
                cache_batch.cache_dict[k] = torch.cat([cache_batch.cache_dict[k], v], dim=0)
    return torch.cat(logits_batch, dim=0), cache_batch

In [12]:
def to_first_token_ids(strings):
    """
    Small function that returns the first token id of strings
    Input:
    - strings: List[str]
    Output:
    - ids: List[int]

    Example:
    ["spam", "not spam"] -> ["spam", "not "] (first tokens) -> [3241, 124]
    """
    ids = set()
    for s in strings:
        toks = model.tokenizer.encode(s, add_special_tokens=False)
        if toks:
            ids.add(toks[0])
    return torch.tensor(sorted(ids), dtype=torch.int)

if model.tokenizer.pad_token is None:
    model.tokenizer.pad_token = model.tokenizer.eos_token
model.tokenizer.padding_side = "left"  # so final position is always index -1

In [13]:
correct_indices = to_first_token_ids(ANSWER_STRINGS[TASK_TYPE])
incorrect_indices = to_first_token_ids(ANSWER_STRINGS[INJECTED_TASK])

In [14]:
def get_logit_diff(logits, correct_indices=correct_indices, incorrect_indices=incorrect_indices):
    if len(logits.shape) == 3:
        # Get final logits only
        logits = logits[:, -1, :]
    correct_logits = logits.gather(1, correct_indices.unsqueeze(0))
    incorrect_logits = logits.gather(1, incorrect_indices.unsqueeze(0))
    return (correct_logits - incorrect_logits).mean()

clean_logits, _ = run_all_prompts_with_cache(clean)
corrupted_logits, _ = run_all_prompts_with_cache(corrupted)

clean_logits = clean_logits.to('cpu')
corrupted_logits = corrupted_logits.to('cpu')

clean_logit_diff = get_logit_diff(clean_logits.to('cpu'), correct_indices, incorrect_indices).item()
print(f"Clean logit diff: {clean_logit_diff:.4f}")

corrupted_logit_diff = get_logit_diff(corrupted_logits.to('cpu'), correct_indices, incorrect_indices).item()
print(f"Corrupted logit diff: {corrupted_logit_diff:.4f}")

100%|██████████| 2/2 [00:00<00:00,  4.90it/s]

Clean logit diff: 3.6900
Corrupted logit diff: -5.1882


In [15]:
CLEAN_BASELINE = clean_logit_diff
CORRUPTED_BASELINE = corrupted_logit_diff


def recovery_metric(logits, correct_indices=correct_indices, incorrect_indices=incorrect_indices):
    return (get_logit_diff(logits, correct_indices, incorrect_indices) - CORRUPTED_BASELINE) / (
        CLEAN_BASELINE - CORRUPTED_BASELINE
    )

In [16]:
# NBVAL_IGNORE_OUTPUT
filter_hook_z = lambda name:"_z" in name


def _fix_cache_device_and_pos_embed(cache, device):
    """Move run_with_cache tensors back to model device (Bug 5 workaround)
    and expand pos_embed to batch dimension (Bug 3 workaround)."""
    for key in list(cache.cache_dict.keys()):
        val = cache.cache_dict[key]
        if isinstance(val, torch.Tensor):
            cache.cache_dict[key] = val.to(device)

    # Expand pos_embed from [1, seq, d_model] to [batch, seq, d_model]
    if "hook_pos_embed" in cache.cache_dict:
        pe = cache.cache_dict["hook_pos_embed"]
        if pe.shape[0] == 1:
            # Infer batch size from any other cached activation
            for k, v in cache.cache_dict.items():
                if k != "hook_pos_embed" and isinstance(v, torch.Tensor) and v.ndim >= 2 and v.shape[0] > 1:
                    cache.cache_dict["hook_pos_embed"] = pe.expand(v.shape[0], -1, -1)
                    break


def _fix_head_result_cache(cache, model, is_grad=False):
    """Fix hook_result shape for TransformerBridge compatibility.

    TransformerBridge's hook_result captures [batch, pos, d_model] (post-head-sum)
    but stack_head_results expects [batch, pos, n_heads, d_model] (pre-sum).

    For forward caches: remove entries so compute_head_results recomputes from z + W_O.
    For grad caches: expand gradient from [batch, pos, d_model] to
    [batch, pos, n_heads, d_model] since d(loss)/d(result_h) = d(loss)/d(out) for all h.
    """
    n_heads = model.cfg.n_heads
    for layer in range(model.cfg.n_layers):
        key = f"blocks.{layer}.attn.hook_result"
        if key not in cache.cache_dict:
            continue
        val = cache.cache_dict[key]
        if not isinstance(val, torch.Tensor) or val.ndim != 3:
            continue  # Already correct shape or not a tensor
        if is_grad:
            # Gradient of sum is identity: d(loss)/d(result_h) = d(loss)/d(out)
            cache.cache_dict[key] = val.unsqueeze(-2).expand(
                *val.shape[:-1], n_heads, val.shape[-1]
            )
        else:
            # Remove so compute_head_results can recompute from z + W_O
            del cache.cache_dict[key]


def get_cache_fwd_and_bwd(model, prompts, metric):
    model.reset_hooks()
    device = model.cfg.device
    grad_cache = {}

    def backward_cache_hook(act, hook):
        grad_cache[hook.name] = act.detach()

    model.add_hook(filter_hook_z, backward_cache_hook, "bwd")

    # run_with_cache handles forward caching with proper alias resolution
    output, fwd_cache = run_all_prompts_with_cache(prompts, names_filter=filter_hook_z)

    # Fix forward cache: move from CPU back to model device, expand pos_embed
    _fix_cache_device_and_pos_embed(fwd_cache, device)

    value = metric(output.to('cpu'))
    value.backward()
    model.reset_hooks()

    # Add alias entries for grad_cache (backward hooks use hook.name, not dict key).
    # Only add aliases that pass our filter to avoid KeyErrors downstream.
    for key, hp in model.hook_dict.items():
        if hp.name != key and filter_not_qkv_input(key):
            if hp.name in grad_cache and key not in grad_cache:
                grad_cache[key] = grad_cache[hp.name]

    # Fix grad cache pos_embed batch dimension too
    grad_act_cache = ActivationCache(grad_cache, model)
    _fix_cache_device_and_pos_embed(grad_act_cache, device)

    # Fix hook_result shape: TransformerBridge captures [batch, pos, d_model]
    # but stack_head_results expects [batch, pos, n_heads, d_model]
    _fix_head_result_cache(fwd_cache, model, is_grad=False)
    _fix_head_result_cache(grad_act_cache, model, is_grad=True)

    return (
        value.item(),
        fwd_cache,
        grad_act_cache,
    )


In [17]:
model.cfg.use_attn_result = False
model.cfg.use_attn_in = False
model.cfg.use_hook_mlp_in = False

In [18]:
clean_value, clean_cache, clean_grad_cache = get_cache_fwd_and_bwd(
    model, clean, recovery_metric
)

print("Clean Value:", clean_value)
print("Clean Activations Cached:", len(clean_cache))
print("Clean Gradients Cached:", len(clean_grad_cache))
assert abs(clean_value - 1.0) < 1e-5, f"Expected 1.0, got {clean_value}"

corrupted_value, corrupted_cache, corrupted_grad_cache = get_cache_fwd_and_bwd(
    model, corrupted, recovery_metric
)
print("Corrupted Value:", corrupted_value)
print("Corrupted Activations Cached:", len(corrupted_cache))
print("Corrupted Gradients Cached:", len(corrupted_grad_cache))
assert abs(corrupted_value) < 1e-5, f"Expected 0.0, got {corrupted_value}"

100%|██████████| 2/2 [00:00<00:00,  6.42it/s]
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:330.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


Clean Value: 1.0
Clean Activations Cached: 28
Clean Gradients Cached: 28


100%|██████████| 2/2 [00:00<00:00,  6.76it/s]


Corrupted Value: 0.0
Corrupted Activations Cached: 28
Corrupted Gradients Cached: 28


In [23]:
print(clean_cache.cache_dict['blocks.0.attn.hook_z'].shape)

torch.Size([2, 95, 12, 128])


In [24]:
print(corrupted_cache.cache_dict['blocks.0.attn.hook_z'].shape)

torch.Size([2, 104, 12, 128])


In [22]:
from typing_extensions import Literal

HEAD_NAMES = [
    f"L{l}H{h}" for l in range(model.cfg.n_layers) for h in range(model.cfg.n_heads)
]

def stack_head_vector_from_cache(
    cache, activation_name: Literal["q", "k", "v", "z"]
):
    """Stacks the head vectors from the cache from a specific activation (key, query, value or mixed_value (z)) into a single tensor."""
    stacked_head_vectors = torch.stack(
        [cache[activation_name, l] for l in range(model.cfg.n_layers)], dim=0
    )
    stacked_head_vectors = einops.rearrange(
        stacked_head_vectors,
        "layer batch pos head_index d_head -> (layer head_index) batch pos d_head",
    )
    return stacked_head_vectors


def attr_patch_head_vector(
    clean_cache: ActivationCache,
    corrupted_cache: ActivationCache,
    corrupted_grad_cache: ActivationCache,
    activation_name: Literal["q", "k", "v", "z"],
):
    labels = HEAD_NAMES

    clean_head_vector = stack_head_vector_from_cache(clean_cache, activation_name)
    print(clean_head_vector.shape)
    corrupted_head_vector = stack_head_vector_from_cache(
        corrupted_cache, activation_name
    )
    print(corrupted_head_vector.shape)
    corrupted_grad_head_vector = stack_head_vector_from_cache(
        corrupted_grad_cache, activation_name
    )
    print(corrupted_grad_head_vector.shape)
    head_vector_attr = einops.reduce(
        corrupted_grad_head_vector * (clean_head_vector - corrupted_head_vector),
        "component batch pos d_head -> component pos",
        "sum",
    )
    return head_vector_attr, labels


head_vector_attr_dict = {}
for activation_name, activation_name_full in [
    ("z", "Mixed Value"),
]:
    display(f"#### {activation_name_full} Head Vector Attribution Patching")
    head_vector_attr_dict[activation_name], head_vector_labels = attr_patch_head_vector(
        clean_cache, corrupted_cache, corrupted_grad_cache, activation_name
    )
    imshow(
        head_vector_attr_dict[activation_name],
        y=head_vector_labels,
        yaxis="Component",
        xaxis="Position",
        title=f"{activation_name_full} Attribution Patching",
    )
    sum_head_vector_attr = einops.reduce(
        head_vector_attr_dict[activation_name],
        "(layer head) pos -> layer head",
        "sum",
        layer=model.cfg.n_layers,
        head=model.cfg.n_heads,
    )
    imshow(
        sum_head_vector_attr,
        yaxis="Layer",
        xaxis="Head Index",
        title=f"{activation_name_full} Attribution Patching Sum Over Pos",
    )

'#### Mixed Value Head Vector Attribution Patching'

torch.Size([336, 2, 95, 128])
torch.Size([336, 2, 104, 128])
torch.Size([336, 1, 104, 128])


RuntimeError: The size of tensor a (95) must match the size of tensor b (104) at non-singleton dimension 2